In [1]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier
from scipy.stats import uniform
import pandas as pd

In [3]:
dataPath = 'datasets/harus/'

xdata = pd.read_csv(dataPath + "xdata.csv", sep=";")                     
ydata = pd.read_csv(dataPath + "ydata.csv", sep=";")

xtrain, xvaltest, ytrain, yvaltest = train_test_split( 
    xdata,
    ydata,
    random_state=0,
    train_size=0.66,
    stratify=ydata                                  # Preserve label imbalance across train- and test datasets
)

xval, xtest, yval, ytest = train_test_split( 
    xvaltest,
    yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=yvaltest                                  # Preserve label imbalance across train- and test datasets
)

ev_val = [(xval,yval)]
ev_all = [(xtrain,ytrain),(xval,yval),(xtest,ytest)]


In [ ]:


param_dist = {
    'max_depth': [1,2,3,4,5,6,7,8,9,10],
    'n_estimators': [100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500],
    'min_child_weight': [1,2,3,4,5,6,7,8,9,10],
    'subsample': uniform(0.4, 0.6),
    'colsample_bytree': uniform(0.1, 0.9),
    'learning_rate': uniform(0.01, 0.29)
}

xgb = XGBClassifier(
    objective='multi:softmax',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10
)

random_search = RandomizedSearchCV(
    estimator=xgb, 
    param_distributions=param_dist, 
    scoring='accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
random_search.fit(xtrain, ytrain,
                  eval_set = ev_val,
                  verbose=False)

# Print best parameters
print(f"Best parameters: {random_search.best_params_}")
print(f"Best score: {random_search.best_score_}")
results = pd.DataFrame(random_search.cv_results_)
results.to_csv("harus_random_search_results.csv", index=False)

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Best parameters: {'colsample_bytree': np.float64(0.9728281609720739), 'learning_rate': np.float64(0.19941061038140193), 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 1500, 'subsample': np.float64(0.7646984012292807)}
Best score: 0.9897012292775831


# Mögliche Ranges

Gem. Paper: Performance Comparison of Grid Search and Random Search Methods for Hyperparameter Tuning in Extreme Gradient Boosting Algorithm to Predict Chronic Kidney Failure (dimas-2021)

|Parameter|Range|in Code|
|---|---|---|
|max_depth|[1,3,5]|[1,3,5] stats.randint(1,6)
|learning_rate|[0.1,0.3]|uniform(0.1,0.2)
|n_estimators|[?,?]|---|
|gamma|[0.1, …, 0.9]|uniform(0.1,0.8)|
|subsample|[0.1,...,0.9]|uniform(0.1,0.8)|
|colsample_bytree|[0.1,...,0.9]|uniform(0.1,0.8)|

scipy.stats.randint(low, high) ist das Pendant zu uniform für Ganzzahlen: [low, high-1]

|Parameter|Range|in Code|
|---|---|---|
|max_depth|[1,3,5]|[1,3,5]|
|learning_rate|[0.1,0.3]|uniform(0.1,0.2)
|n_estimators|[?,?]|---|
|gamma|[0.1, …, 0.9]|uniform(0.1,0.8)|
|subsample|[0.1,...,0.9]|uniform(0.1,0.8)|
|colsample_bytree|[0.1,...,0.9]|uniform(0.1,0.8)|

# Hyperparams und was sie machen

- max_depth: Legt fest, wie tief ein Baum werden darf
- n_estimators: Maximale Anzahl an Schätzern im Ensemble (bei Multiclass ist die Anzahl an Schätzern n_estimators * Anzahl der Klassen)
- Learning_rate $\eta$: Kontrolliert die Geschwindigkeit, mit welcher die Gewichtungen mit jedem neuen Schätzer/pro Iteration angepasst werden (default 0.3) -> Je kleiner der Wert, desto langsamer findet die Anpassung statt 
- Gamma $\gamma$: Bestimmt einen Wert, um den der Modellverlust durch die Aufspaltung eines Knotens sinken muss, um diesen zu rechtfertigen. Nachdem ein Baum erzeugt worden ist, bestimmt der Gamma-Wert, welche Knoten entfernt bzw. wieder zusammengeführt werden. Dieser Prozess wird auch als Pruning bezeichnet. (default: 0)
- Lambda $\lambda$: L2 Regulierung - je höher der Wert, desto simpler wird das Modell (default: 1)
- Subsample: Bestimmt den prozentualen Anteil der Instanzen aus dem Datensatz, die verwendet werden dürfen, um daraus einen Schätzer zu trainieren
- Colsample_bytree: Bestimmt den prozentualen Anteil der Spalten/Merkmalen aus dem Datensatz, die verwendet werden dürfen, um daraus einen Schätzer zu trainieren

### Empfehlung für die Auswahl an Hyperparametern (XGBoosting): 

- max_depth & min_child_weight: Kontrollieren die maximale Tiefe der einzelnen Bäume (Hilfreich um die richtige Balance aus Komplexität und Generalisierung zu treffen)
- Subsample und colsample_bytree: Subsample gibt an, welcher Anteil an Instanzen aus dem Datensatz pro Baum verwendet werden darf - colsample_bytree gibt an welcher Anteil an Spalten/Merkmalen pro Baum verwendet werden darf
- learning_rate: Kontrolliert die Geschwindigkeit, mit welcher die Gewichtungen mit jedem neuen Schätzer/pro Iteration angepasst werden (default 0.3) -> Je kleiner der Wert, desto langsamer findet die Anpassung statt 

### Empfehlung Ranges (XGBoosting)

{
    
    'max_depth': [3, 5, 7, 9],

    'min_child_weight': [1, 3, 5, 7],

    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],

    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],

    'learning_rate': [0.01, 0.05, 0.1, 0.2]

}

$F_{t}(x) = F_{t-1}(x) + \eta \cdot h_{t}(x)$

Hierbei gilt:

$F_{t}(x)$ ist die aktualisierte Prädiktion für ein gegebenes Beispiel ( x ) in der t-ten Iteration.

$F_{t-1}(x)$ ist die Prädiktion aus der vorherigen Iteration (t-1).

$\eta$ ist die learning_rate (Eta), die den Einfluss des neuen Baums steuert.

$h_{t}(x)$ ist der in der t-ten Iteration hinzugefügte Baum, der die Residuen (Fehler) der vorherigen Bäume approximiert.

Der learning_rate wirkt als Dämpfungsfaktor, der den Einfluss des neuen Baums auf das Gesamtmodell verringert. Dadurch können kleinere Änderungen an den Prädiktionen vorgenommen und Überanpassungen vermieden werden, was zu einer robusteren Modellleistung führt.

In [ ]:
param_dist2 = {
    'max_depth': [1,3,5,7,9],
    'n_estimators': [100,200,300,400,500,600,700,800,900,1000],
    'min_child_weight': [1,3,5,7],
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'learning_rate': uniform(0.01, 0.29),
    'gamma': uniform(0.1, 0.8)
}

xgb2 = XGBClassifier(
    objective='multi:softmax',
    tree_method='exact',
    n_jobs=-1,
    early_stopping_rounds=10
)

random_search2 = RandomizedSearchCV(
    estimator=xgb2, 
    param_distributions=param_dist2, 
    scoring='accuracy',
    n_iter=250, 
    cv=5,
    n_jobs=-1, 
    verbose=1, 
    random_state=0
)
random_search2.fit(xtrain, ytrain,
                  eval_set = ev_val,
                  verbose=False)


Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Best parameters: {'colsample_bytree': np.float64(0.8582280977824015), 'gamma': np.float64(0.12828994860439275), 'learning_rate': np.float64(0.13481670745733776), 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 1000, 'subsample': np.float64(0.6575317462724224)}
Best score: 0.987641864692897


In [8]:

# Print best parameters
print(f"Best parameters: {random_search2.best_params_}")
print(f"Best score: {random_search2.best_score_}")
results2 = pd.DataFrame(random_search2.cv_results_)
results2.to_csv("harus_random_search_results.csv", index=False)

Best parameters: {'colsample_bytree': np.float64(0.8582280977824015), 'gamma': np.float64(0.12828994860439275), 'learning_rate': np.float64(0.13481670745733776), 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 1000, 'subsample': np.float64(0.6575317462724224)}
Best score: 0.987641864692897


In [9]:
stringToText = "Best parameters: " + str(random_search2.best_params_) + "\n"
stringToText += "Best score: " + str(random_search2.best_score_) + "\n"
with open("harus-params.txt", "w", encoding="utf-8") as file:
    # Schreibe den String in die Datei
    file.write(stringToText)
